# Лаборатори 11. Eduge мэдээ ангилал ба бичвэрийн давхцал

Eduge корпусыг Bolorsoft нийтэлсэн. [Эх өгөгдлийн тайлбар](https://github.com/tugstugi/mongolian-nlp) болон [анхны дасгал](https://github.com/sharavsambuu/mongolian-text-classification). Eduge CSV, SentencePiece-ийн cased загвар болон нэмэлт орчуулгын корпусыг хичээлийн textclassification.zip архиваас уншина.

In [1]:
from pathlib import Path
from shutil import copyfileobj
from tempfile import NamedTemporaryFile
from zipfile import ZipFile

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "textclassification.zip").is_file())
WORK = ROOT / ".work" / "lab11_12"

def eduge_file(name):
    member = f"textclassification/{name}"
    path = WORK / member
    with ZipFile(ROOT / "textclassification.zip") as archive:
        size = archive.getinfo(member).file_size
        if path.is_file() and path.stat().st_size == size:
            return path
        path.parent.mkdir(parents=True, exist_ok=True)
        with NamedTemporaryFile(dir=path.parent, delete=False) as temporary:
            temp_path = Path(temporary.name)
            try:
                with archive.open(member) as source:
                    copyfileobj(source, temporary)
                temporary.close()
                temp_path.replace(path)
            finally:
                temp_path.unlink(missing_ok=True)
    return path

In [2]:
import re
import time
import unicodedata

import pandas as pd
import sentencepiece as spm
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

sp = spm.SentencePieceProcessor(model_file=str(eduge_file("mn_cased.model")))
def sp_tokenize(text):
    return sp.encode(text, out_type=str)

print(" ".join(sp_tokenize("Мөнгөө тушаачихсаныхаа дараа мэдэгдээрэй".lower())))

▁мөнгөө ▁тушаа чихсан ыхаа ▁дараа ▁мэдэгд ээрэй


## 1. Eduge мэдээний ангилал

In [3]:
df = pd.read_csv(eduge_file("eduge.csv"))
df.columns = df.columns.str.strip()
train, test = train_test_split(df, test_size=0.1, random_state=999, stratify=df["label"])
print(f"Нийт={len(df):,}, train={len(train):,}, test={len(test):,}")
display(pd.DataFrame({"train": train["label"].value_counts(), "test": test["label"].value_counts()}))
assert len(df) == len(train) + len(test) and len(df["label"].unique()) == 9
unmatched = ~test["news"].isin(train["news"])
print(f"Сургалтын олонлогт яг ижил бичвэртэй тестийн мэдээ: {(~unmatched).sum():,}")
print(f"Яг ижил бичвэргүй тестийн мэдээ: {unmatched.sum():,}")

Нийт=75,661, train=68,094, test=7,567


,train,test
label,,
спорт,12088,1343
улс төр,12065,1341
эдийн засаг,11152,1239
хууль,7455,829
урлаг соёл,6590,732
эрүүл мэнд,5200,578
технологи,4972,553
байгал орчин,4547,505
боловсрол,4025,447


Сургалтын олонлогт яг ижил бичвэртэй тестийн мэдээ: 293
Яг ижил бичвэргүй тестийн мэдээ: 7,274


In [4]:
models = {
    "SentencePiece + TF-IDF + SVM": Pipeline([
        ("vect", CountVectorizer(tokenizer=sp_tokenize, token_pattern=None, lowercase=True)),
        ("tfidf", TfidfTransformer()),
        ("clf", SGDClassifier(loss="hinge", penalty="l2", alpha=1e-4, random_state=0)),
    ]),
    "Word (1, 2)-grams + TF-IDF + SVM": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=400_000, sublinear_tf=True)),
        ("clf", SGDClassifier(loss="hinge", penalty="l2", alpha=1e-5, random_state=0)),
    ]),
}

results = []
unmatched_results = []
for name, model in models.items():
    started = time.perf_counter()
    model.fit(train["news"], train["label"])
    train_seconds = time.perf_counter() - started
    started = time.perf_counter()
    prediction = model.predict(test["news"])
    results.append({
        "model": name, "train_n": len(train), "test_n": len(test),
        "features": len(model.named_steps["vect"].vocabulary_),
        "accuracy": accuracy_score(test["label"], prediction),
        "macro_f1": f1_score(test["label"], prediction, average="macro"),
        "train_s": train_seconds, "predict_s": time.perf_counter() - started,
    })
    unmatched_results.append({
        "model": name, "test_n": unmatched.sum(),
        "accuracy": accuracy_score(test.loc[unmatched, "label"], prediction[unmatched]),
        "macro_f1": f1_score(test.loc[unmatched, "label"], prediction[unmatched], average="macro"),
    })
    if name == next(iter(models)):
        print(classification_report(test["label"], prediction, digits=4))
print("Анхны 90/10 хуваалтын үр дүн")
display(pd.DataFrame(results).set_index("model").round(4))
print("Сургалттай үгчлэн давхацсан тестийн мэдээг хассан үнэлгээ")
display(pd.DataFrame(unmatched_results).set_index("model").round(4))
text_clf = next(iter(models.values()))

              precision    recall  f1-score   support

байгал орчин     0.9000    0.8198    0.8580       505
   боловсрол     0.8735    0.8031    0.8368       447
       спорт     0.9705    0.9806    0.9756      1343
   технологи     0.9084    0.9150    0.9117       553
     улс төр     0.9504    0.8867    0.9174      1341
  урлаг соёл     0.9051    0.9508    0.9274       732
       хууль     0.8663    0.8830    0.8746       829
 эдийн засаг     0.8563    0.9282    0.8908      1239
  эрүүл мэнд     0.9199    0.9135    0.9167       578

    accuracy                         0.9107      7567
   macro avg     0.9056    0.8979    0.9010      7567
weighted avg     0.9117    0.9107    0.9104      7567



Анхны 90/10 хуваалтын үр дүн


,train_n,test_n,features,accuracy,macro_f1,train_s,predict_s
model,,,,,,,
SentencePiece + TF-IDF + SVM,68094,7567,25384,0.9107,0.9010,15.6662,1.5521
"Word (1, 2)-grams + TF-IDF + SVM",68094,7567,400000,0.9179,0.9085,37.8868,1.8494


Сургалттай үгчлэн давхацсан тестийн мэдээг хассан үнэлгээ


,test_n,accuracy,macro_f1
model,,,
SentencePiece + TF-IDF + SVM,7274,0.9131,0.9026
"Word (1, 2)-grams + TF-IDF + SVM",7274,0.9221,0.9118


Анхны stratified 90/10 хуваалтын тестийн 7,567 мэдээнээс 293 нь сургалтын олонлогт үгчлэн давхцсан. Давхцаагүй 7,274 мэдээнд тусад нь үнэлэхэд SentencePiece загварын accuracy 91.31%, macro-F1 0.9026, үгийн 1–2 грамтай загварын accuracy 92.21%, macro-F1 0.9118 байв. Энэ дэд олонлог дээр хоёр загварын оноо бага зэрэг өссөн. Иймээс энэ хуваалт дээр үгчлэн давхцсан мэдээнүүдийг оруулснаар оноо өсөөгүй. Үгийн 1–2 грамтай загвар хоёр үнэлгээнд өндөр оноо авсан бөгөөд 400,000 шинж хэрэглэж, сургалт нь удаан байв.

### Эх notebook-ийн GoGo жишээнүүд

In [5]:
ex_text_str1 = "FIBA-ын 2025 оны Азийн аварга шалгаруулах тэмцээний урьдчилсан нэгдүгээр шатны D хэсгийн тоглолтууд Улаанбаатар хотод болж өндөрлөлөө. Таван өдөр үргэлжилсэн хэсгийн тоглолтууд дуусаж хэсгийн тэргүүн багууд тодров. Өнөөдөр Монголын шигшээ баг болон Тайландын шигшээ багууд хэсгийн аваргын төлөө тоглоход Тайландын шигшээ баг 109-105 харьцаагаар хожлоо. Монголын шигшээ баг тоглолтын эхний хагаст 10 онооны зөрүүгээр тэргүүлж байсан ч гуравдугаар үед Тайланд илүү тоглож оноо барьсан. Улмаар дөрөвдүгээр үед Тайланд тэргүүллээ баталгаажуулж багийн гол тоглогч Фредерих Лишийн дараалсан холын шидэлтүүд байгаа оносноор Монголчууд урьдчилсан шатанд анх удаа хожигдлоо. Энэ тоглолтын өмнө Вьетнам, Малайзын шигшээ багууд хэн нь хэсгийн гуравдугаар байраар дараагийн шатанд шалгарахаа шийдэж тоглоход Малайз 73-56 харьцаагаар хожлоо. Урьдчилсан нэгдүгээр шатны С хэсгээс Гуам 4, Хонконг 3, Сингапур 2 оноотой хоёрдугаар шатанд шалгарсан бол D хэсгээс ийнхүү Тайланд, Монгол, Малайз шалгарав. Энэ зургаан баг урьдчилсан хоёрдугаар шатанд 2023 оны хоёрдугаар сард тоглоно. Үүнээс хамгийн өндөр оноо авсан 4 баг нь Азийн аваргын урьдчилсан гуравдугаар шатанд шалгарч тоглох учиртай."

ex_text_str2 = "Samsung Experience Store-ийн тохижилт, үйлчилгээний стандартыг Samsung брэндийн бодлогын дагуу хийж гүйцэтгэсэн байна. Sain electronics компани нь Samsung Experience store-оор дамжуулан Samsung брэндийн Албан ёсны гар утас, телевиз, тоос сорогч, цахилгаан бараа болон дагалдах хэрэгслийг хэрэглэгчдэдээ нэг дороос үзэж, сонгон, худалдан авалт хийх боломжоор хангаж байгаа анхны компани юм"

ex_text_str3 = "FIBA-ын 2025 оны Азийн аварга шалгаруулах тэмцээний урьдчилсан нэгдүгээр шатны D хэсгийн тоглолтууд Улаанбаатар хотод дөрөв дэх өдрөө үргэлжилж байна. Монголын шигшээ баг өнөөдөр гурав дахь тоглолтоо Вьетнам улсын шигшээ багтай хийсэн бөгөөд 88-77 харьцаагаар хожлоо. Шигшээ баг уржигдар Малайзыг хүндхэн тоглолтын эцэст 94-92 харьцаагаар, өчигдөр Таитиг 62-91 харьцаагаар илүүрхэн хожоод байв. Ийнхүү өнөөдөр хэсэгтээ гурван хожилтой болж дараагийн шатанд шалгаран тоглох болзлоо хангалаа. Тэд маргааш хэсгийг тэргүүлэгч Тайландын шигшээ багтай тоглож хэсгийн хэддүгээр байраар шалгарахаа шийднэ."
demo_articles = [ex_text_str1, ex_text_str2, ex_text_str3]
display(pd.DataFrame({
    "жишээ": ["FIBA: Тайланд", "Samsung Experience Store", "FIBA: Вьетнам"],
    "таамаг": text_clf.predict(demo_articles),
}))

,жишээ,таамаг
0,FIBA: Тайланд,спорт
1,Samsung Experience Store,технологи
2,FIBA: Вьетнам,спорт


### Бодлого 1. Вэбээс авсан мэдээ ангилах

GoGo-ийн 2026 оны мэдээнүүдийн агуулгыг товчлон загварт оруулав. [3×3 сагсан бөмбөг](https://gogo.mn/r/q994ym), [Монголбанкны бодлогын хүү](https://gogo.mn/r/9ll8og).

In [6]:
web_articles = pd.DataFrame([
    {
        "news": "Азийн наадмын 3х3 сагсан бөмбөгийн хэсгийн тоглолтод Монголын эрэгтэй шигшээ Казахстаны багийг 13:8-аар хожиж анхны ялалтаа байгууллаа.",
        "expected": "спорт",
    },
    {
        "news": "Монголбанк бодлогын хүүгээ 12.5 хувьд хадгаллаа. Инфляц өндөр байгаа тул бодлогын хүүг бууруулах боломж хязгаарлагдмал гэж үзжээ.",
        "expected": "эдийн засаг",
    },
])
for name, model in models.items():
    web_articles[name] = model.predict(web_articles["news"])
display(web_articles)

,news,expected,SentencePiece + TF-IDF + SVM,"Word (1, 2)-grams + TF-IDF + SVM"
0,Азийн наадмын 3х3 сагсан бөмбөгийн хэсгийн тог...,спорт,спорт,спорт
1,Монголбанк бодлогын хүүгээ 12.5 хувьд хадгалла...,эдийн засаг,эдийн засаг,эдийн засаг


## 2. Бодлого 2. Бичвэрийн давхцлыг илрүүлэх

Эх жишээний fit(train) нь DataFrame-ийн баганын нэрийг уншдаг. Энд нийтлэлийн текст дээр сургасан TF-IDF хувиргалтыг ашиглан анхны гурван жишээний cosine зайг тооцов. Өөрийн аргаар NFC нормчилж, том жижиг үсэг, цэг таслалыг үл харгалзан, сэжигтэй бичвэрийн давтагдаагүй гурван үгт дарааллын хэдэн хувь нь эх бичвэрт байгааг тооцов. Ингэснээр урт эхээс хуулсан богино хэсгийг илрүүлж болно. 0.60 босгыг доорх хяналтын жишээнд ашиглав.

In [7]:
def word_shingles(text, width=3):
    words = re.findall(r"[^\W_]+", unicodedata.normalize("NFC", text).casefold())
    return {tuple(words[i:i + width]) for i in range(len(words) - width + 1)}

def copied_fraction(source, suspect):
    suspect_shingles = word_shingles(suspect)
    if not suspect_shingles:
        return None
    return len(word_shingles(source) & suspect_shingles) / len(suspect_shingles)

vectorizer = text_clf[:-1]
demo_vectors = vectorizer.transform(demo_articles)
for i, j in [(0, 1), (1, 2), (0, 2)]:
    distance = 1 - cosine_similarity(demo_vectors[i], demo_vectors[j]).item()
    print(f"GoGo {i + 1}/{j + 1}: cosine distance={distance:.3f}")

source = demo_articles[0]
examples = [
    ("шууд хуулсан", source.split(". ")[0], True),
    ("үсэг ба тэмдэг өөрчилсөн", source.split(". ")[0].upper() + "!!!", True),
    ("ижил сэдэв, өөр мэдээ", demo_articles[2], False),
    ("өөр сэдэв", demo_articles[1], False),
    ("хоёр үгтэй", "Монголын баг", None),
]
checks = []
for name, suspect, expected in examples:
    overlap = copied_fraction(source, suspect)
    flag = overlap >= 0.60 if overlap is not None else None
    cosine = cosine_similarity(demo_vectors[0], vectorizer.transform([suspect])).item()
    checks.append((name, cosine, overlap, flag, expected))
    assert flag == expected, name
display(pd.DataFrame(checks, columns=["жишээ", "cosine similarity", "3-gram", "flag", "expected"]).round(3))

GoGo 1/2: cosine distance=0.978
GoGo 2/3: cosine distance=0.990
GoGo 1/3: cosine distance=0.512


,жишээ,cosine similarity,3-gram,flag,expected
0,шууд хуулсан,0.388,1.00,True,True
1,үсэг ба тэмдэг өөрчилсөн,0.361,1.00,True,True
2,"ижил сэдэв, өөр мэдээ",0.488,0.19,False,False
3,өөр сэдэв,0.022,0.00,False,False
4,хоёр үгтэй,0.200,NaN,None,None


Утгыг хадгалж, үгийг сольж бичсэн хэсэг гурван үгт дарааллын шалгалтаас мултарч болно. Өндөр давхцал нь эх сурвалжийг гар аргаар шалгах дохио болно.

## 3. Нэмэлт. Монгол–англи орчуулгын корпус

mn_en.txt доторх +++++SEP+++++ тэмдэгтээр тусгаарласан мөрүүдийг уншиж, 80/20 харьцаагаар хуваав. Жишээ болгон сургалтын 25,000 хосын монгол талд үсгийн n-gram TF-IDF сургаж, шалгах өгүүлбэрт хамгийн ойр өгүүлбэрийн англи орчуулгыг санал болгов. Энэ арга сургалтын санд байгаа орчуулгыг л буцаана.

In [8]:
translation_pairs = []
with eduge_file("mn_en.txt").open(encoding="utf-8", errors="replace") as stream:
    for line in stream:
        fields = line.strip().split("+++++SEP+++++")
        if len(fields) == 2 and all(field.strip() for field in fields):
            translation_pairs.append(tuple(field.strip() for field in fields))
parallel_train, parallel_val = train_test_split(translation_pairs, test_size=0.20, random_state=42)
print(f"Хос өгүүлбэр: нийт={len(translation_pairs):,}; train={len(parallel_train):,}; val={len(parallel_val):,}")
print("MN:", parallel_train[0][0])
print("EN:", parallel_train[0][1])

lookup = dict(parallel_train[:25_000])
retriever = TfidfVectorizer(analyzer="char", ngram_range=(2, 4), min_df=2)
source_mn = list(lookup)
retrieval_vectors = retriever.fit_transform(source_mn)
print(f"Хайлтын санд ашигласан ялгаатай монгол өгүүлбэр: {len(source_mn):,}")
seen_mn = {mn for mn, _ in parallel_train}
for query, reference in [(mn, en) for mn, en in parallel_val[:100] if mn not in seen_mn][:3]:
    similarity = cosine_similarity(retriever.transform([query]), retrieval_vectors)[0]
    best = similarity.argmax()
    print(f"\nMN: {query}\nReference EN: {reference}\nRetrieved EN ({similarity[best]:.3f}): {lookup[source_mn[best]]}")

Хос өгүүлбэр: нийт=1,005,810; train=804,648; val=201,162
MN: Мөн үүнээс гадна Цагдаагийн ерөнхий газрын Тусгай хэлтсийн гурван ажилтан "VENUS" бааранд нэгэн иргэнийг зодож амь насыг нь хэрэг мөн гарсан.
EN: In addition, three employees of the Special Division of the General Police Department were beaten by a citizen in "VENUS" bars and died.


Хайлтын санд ашигласан ялгаатай монгол өгүүлбэр: 25,000



MN: 2013 онд 7.3 тэрбум төгрөг, 2014 онд 10 тэрбум, 2015 онд 13 тэрбум төгрөг төсөвлөсөн байна.
Reference EN: MNT 7.3 billion in 2013, MNT 10 billion in 2014 and MNT 13 billion in 2015.
Retrieved EN (0.464): MNT 380 billion will be spent on salaries, MNT 100 billion will be spent on retirement and MNT 20 billion will be spent on benefits.

MN: Энэ нь юу ч байж болно бүр өчүүхэн юм байсан ч болно (өөрийнхөө дуртай хоолыг идэх, дуртай нэвтрүүлгээ үзэх, ном унших, хүмүүстэй ярих г.м).
Reference EN: It can be anything but small things (eat your favorite foods, enjoy your favorite shows, read books, talk to people, etc.).
Retrieved EN (0.228): So it's a simple and producty product for everyone who loves to read.

MN: Х.Тэмүүжин Үндсэн хуулийн ЦЭЦ гэдэг байгууллага улс төрийн морг гэсэн бол Өнөрбаяр доктор, ЦЭЦ үеэ өнгөрөөсөн нөхдийн цуглуулга гээд хэлчихсэн юм шиг байна лээ.
Reference EN: Kh.Temuujin The Constitutional Court of the Constitutional Court called the political cabaret, Unurbay

Агуулга ойролцоо өгүүлбэр олдлоо ч үйл явдал, нэр, огноо зэрэг нь эх өгүүлбэртэй зөрж болно. Монгол–япон–англи дам орчуулга болон буцаан орчуулгыг анхны notebook нэмэлт туршилтын санаа болгон санал болгосон.